# Практическое занятие 3. Работа со структурированными, полуструктурированными и неструктурированными данными (CSV, JSON, XML)

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p03.ipynb)

**Цели занятия:**

- Различать структурированные, полуструктурированные и неструктурированные данные
- Читать и записывать CSV, JSON и XML средствами Python (`csv`/`pandas`, `json`, `xml.etree.ElementTree`)
- Превращать вложенный JSON и XML в плоские таблицы (`pd.json_normalize`, `pd.read_xml`)
- Извлекать структуру из неструктурированного текста: заголовок, число символов и слов

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Три вида данных

| Вид | Признаки | Примеры | Чем читать в Python |
|---|---|---|---|
| **Структурированные** (structured) | жёсткая схема: строки и столбцы, у каждого столбца один тип | CSV, Excel, таблицы SQL | `pd.read_csv`, `pd.read_excel`, `sqlite3` |
| **Полуструктурированные** (semi-structured) | данные сами описывают себя ключами или тегами; бывают вложенность и необязательные поля | JSON, XML, HTML, JSON Lines, логи | `json`, `xml.etree.ElementTree`, `pd.json_normalize`, `pd.read_xml` |
| **Неструктурированные** (unstructured) | нет заранее заданной схемы | тексты, PDF, изображения, аудио, видео | `open().read()`, регулярные выражения, NLP |

По оценкам аналитиков, 80–90 % всех данных в мире — неструктурированные. Для компьютерного лингвиста это основной материал: корпуса новостей, посты, расшифровки речи. Размеченный корпус (например, в формате TEI XML или CoNLL-U) уже становится полуструктурированным.

### Schema-on-write и schema-on-read

Классическая база данных проверяет схему **при записи** (*schema-on-write*): строку с неверным типом просто не дадут сохранить. В системах Big Data (HDFS, озёра данных, MongoDB) данные часто сохраняют «как есть», а схему накладывают **при чтении** (*schema-on-read*). Это гибко — можно принять JSON с новыми полями, — но вся проверка и приведение типов ложится на код анализа. Именно этим вы займётесь в заданиях: превратите вложенный JSON, XML и текст в аккуратные таблицы с правильными типами.

## CSV — структурированные данные

```python
import pandas as pd

students = pd.read_csv("data/students.csv")              # разделитель по умолчанию — запятая
kl = students[students["faculty"] == "Kompyuter lingvistikasi"]
kl.to_csv("kl_students.csv", index=False, sep=";")       # без столбца индекса, через «;»
back = pd.read_csv("kl_students.csv", sep=";")           # читать нужно с тем же разделителем
```

> **Совет: index=False**  
> Если забыть `index=False`, в файл попадёт индекс DataFrame, а при чтении появится лишний столбец `Unnamed: 0`.

## JSON — модуль json

JSON состоит из объектов `{"ключ": значение}` и массивов `[...]`. В Python они становятся словарями (`dict`) и списками (`list`).

```python
import json

with open("data/books.json", encoding="utf-8") as f:
    books = json.load(f)                 # список словарей

book = books[0]
book["title"]                            # "O'tkan kunlar"
book["publisher"]["city"]                # вложенный объект
book["authors"][0]["name"]               # массив объектов
book.get("isbn", "нет")                  # необязательное поле: get вместо []

with open("out.json", "w", encoding="utf-8") as f:
    json.dump(books, f, ensure_ascii=False, indent=2)
```

Параметр `ensure_ascii=False` сохраняет кириллицу и символы вроде `ʻ` как есть; без него «Мастер» превратится в `\u041c\u0430...`.

## JSON → таблица: json_normalize

```python
df = pd.json_normalize(books)            # publisher.name, price.amount, ... — вложенные поля через точку
reviews = pd.json_normalize(books, record_path="reviews", meta=["id", "title"])
# одна строка на каждый отзыв + столбцы id и title книги
```

Списки внутри записей (`authors`, `reviews`) `json_normalize` по умолчанию оставляет как есть; чтобы развернуть их в строки, укажите `record_path`.

## XML — ElementTree

```xml
<library name="Universitet kutubxonasi">
  <book id="1" lang="uz">
    <title>O'tkan kunlar</title>
    <price currency="UZS">65000</price>
  </book>
</library>
```

```python
import xml.etree.ElementTree as ET

root = ET.parse("data/library.xml").getroot()
root.get("name")                         # атрибут корня
for book in root.findall("book"):        # все дочерние элементы <book>
    book.get("id")                       # атрибут — всегда строка!
    book.findtext("title")               # текст вложенного элемента
    book.find("price").get("currency")   # атрибут вложенного элемента
```

Для простых плоских XML есть `pd.read_xml("data/library.xml")`: каждый `<book>` станет строкой, а атрибуты и дочерние элементы — столбцами (атрибуты вложенных элементов, например `currency`, при этом теряются).

## Текст — неструктурированные данные

```python
with open("data/corpus_uz/uz_01.txt", encoding="utf-8") as f:
    text = f.read()
title = text.split("\n")[0]       # первая строка — заголовок
words = text.split()              # простейшая токенизация по пробелам
len(text), len(words)
```

> **Заметка: Из текста — в таблицу**  
> Типичный конвейер Big Data: собрать тысячи текстов, для каждого вычислить признаки (длина, число слов, язык, тема) и сохранить их в **структурированную** таблицу. Дальше с ней работают как с обычным CSV.

## Задание 1. CSV: чтение, фильтр, запись

1. Прочитайте `data/students.csv` в `students`.
2. Отберите студентов факультета **Kompyuter lingvistikasi** в `kl`.
3. Сохраните `kl` в файл `kl_students.csv` **без индекса** и с разделителем `;`.
4. Прочитайте файл обратно в `kl2` (не забудьте про разделитель) и выведите `kl2.shape`.

<details><summary>💡 Подсказка</summary>

Запись: `kl.to_csv("kl_students.csv", index=False, sep=";")`. Чтение: `pd.read_csv("kl_students.csv", sep=";")`.

</details>

In [ ]:
import pandas as pd

students = ...
kl = ...

# сохраните kl в kl_students.csv

kl2 = ...
print(kl2.shape)
kl2.head()

## Задание 2. JSON: модуль json

Файл `data/books.json` — список из 16 книг со вложенными полями. Используйте модуль `json` (без pandas):

- `books` — загруженный список;
- `titles_uz` — список названий книг на узбекском (`language == "uz"`);
- `no_isbn` — список названий книг, у которых **нет** поля `isbn` (необязательное поле!);
- `n_reviews` — общее число отзывов (`reviews`) по всем книгам;
- сохраните в файл `books_short.json` список словарей `{"title": ..., "language": ...}` для всех книг — с `ensure_ascii=False` и `indent=2`.

<details><summary>💡 Подсказка</summary>

Отсутствие ключа проверяется так: `"isbn" not in b`. Сумма: `sum(len(b.get("reviews", [])) for b in books)`. Запись: `json.dump(short, f, ensure_ascii=False, indent=2)` в файле, открытом с `"w"`.

</details>

In [ ]:
import json

with open("data/books.json", encoding="utf-8") as f:
    books = ...

titles_uz = ...
no_isbn = ...
n_reviews = ...

short = ...
# сохраните short в books_short.json

print(len(books), "книг; на узбекском:", len(titles_uz))
print("Без ISBN:", no_isbn)
print("Отзывов:", n_reviews)

## Задание 3. Вложенный JSON → таблицы (json_normalize)

Список `books` уже загружен. С помощью `pd.json_normalize`:

1. `df` — плоская таблица книг (вложенные поля станут столбцами `publisher.name`, `price.amount`, …).
2. `avg_price` — Series: средняя цена (`price.amount`) по языку книги (`language`).
3. `authors` — таблица авторов: одна строка на каждого автора каждой книги, плюс столбцы `id` и `title` книги (`record_path` и `meta`).
4. `reviews` — таблица отзывов аналогично (`record_path="reviews"`).
5. `mean_stars` — Series: средняя оценка `stars` для каждой книги (`title`).

<details><summary>💡 Подсказка</summary>

`pd.json_normalize(books, record_path="authors", meta=["id", "title"])`. Средние: `groupby("language")["price.amount"].mean()`.

</details>

In [ ]:
# Подготовительный код задания
import json
import pandas as pd
with open("data/books.json", encoding="utf-8") as f:
    books = json.load(f)

In [ ]:
df = ...
print(df.columns.tolist())

avg_price = ...

authors = ...
reviews = ...
mean_stars = ...

print(avg_price)
print(len(authors), "авторов,", len(reviews), "отзывов")
mean_stars.sort_values(ascending=False).head()

## Задание 4. XML: разбор с ElementTree

Разберите каталог `data/library.xml` модулем `xml.etree.ElementTree`:

- `lib_name` — значение атрибута `name` корневого элемента;
- `lib` — DataFrame, одна строка на каждый `<book>`, столбцы: `id`, `lang` (атрибуты), `title`, `author`, `year`, `genre`, `price`, `currency` (атрибут элемента `<price>`), `copies`. Столбцы `id`, `year`, `price`, `copies` должны быть **целыми числами** (в XML всё — строки!);
- `total_copies` — общее число экземпляров;
- `not_available` — список названий книг, у которых `copies == 0`.

<details><summary>💡 Подсказка</summary>

Атрибут: `book.get("id")`, текст: `book.findtext("year")`, атрибут вложенного элемента: `book.find("price").get("currency")`. Не забудьте `int(...)`.

</details>

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

root = ET.parse("data/library.xml").getroot()
lib_name = ...

rows = []
for book in root.findall("book"):
    rows.append({
        "id": ...,
        "lang": book.get("lang"),
        "title": book.findtext("title"),
        # author, year, genre, price, currency, copies
    })

lib = pd.DataFrame(rows)
total_copies = ...
not_available = ...

print(lib_name, "| экземпляров:", total_copies, "| нет в наличии:", not_available)
lib.head()

## Задание 5. read_xml и конвертация форматов

1. Прочитайте `data/library.xml` одной командой `pd.read_xml` в `lib2`.
2. Сохраните `lib2` в `library.csv` (без индекса) и в `library.json` — список записей (`orient="records"`, `force_ascii=False`).
3. `sizes` — словарь размеров трёх файлов в байтах: `{"xml": ..., "csv": ..., "json": ...}` (`os.path.getsize`).
4. `smallest` — ключ самого компактного формата.

Подумайте: почему CSV получился компактнее JSON и XML?

<details><summary>💡 Подсказка</summary>

`lib2.to_json("library.json", orient="records", force_ascii=False)`; минимум по значениям словаря: `min(sizes, key=sizes.get)`.

</details>

In [ ]:
import os
import pandas as pd

lib2 = ...
print(lib2.dtypes)

# сохраните library.csv и library.json

sizes = {
    "xml": os.path.getsize("data/library.xml"),
    "csv": ...,
    "json": ...,
}
smallest = ...
print(sizes, "->", smallest)

## Задание 6. Текст → таблица признаков

Тексты корпуса `data/corpus_uz/uz_01.txt … uz_16.txt` — неструктурированные данные. Превратите их в структурированную таблицу.

1. Прочитайте `data/corpus_uz/uz_14.txt` в строку `text`; `title` — первая строка текста; `n_chars` — число символов; `n_words` — число слов при разбиении `text.split()`.
2. Для **всех** 16 файлов `.txt` из папки соберите DataFrame `corpus` со столбцами `file` (имя файла, например `"uz_01.txt"`), `title`, `n_chars`, `n_words`, упорядоченный по `file`.
3. `longest` — заголовок текста с наибольшим числом слов.

Файл `meta.csv` в той же папке — не текст корпуса, его нужно пропустить.

<details><summary>💡 Подсказка</summary>

Первая строка: `text.split("\n")[0]`. В цикле пропускайте файлы, которые не заканчиваются на `.txt`: `if not name.endswith(".txt"): continue`. Самый длинный: `corpus.loc[corpus["n_words"].idxmax(), "title"]`.

</details>

In [ ]:
import os
import pandas as pd

with open("data/corpus_uz/uz_14.txt", encoding="utf-8") as f:
    text = ...
title = ...
n_chars = ...
n_words = ...
print(title, n_chars, n_words)

rows = []
for name in sorted(os.listdir("data/corpus_uz")):
    ...

corpus = pd.DataFrame(rows)
longest = ...
print("Самый длинный:", longest)
corpus

## Задание 7. Классификатор файлов по виду данных

Напишите функцию `classify(filename)`, которая по **расширению** файла (регистр не важен) возвращает вид данных:

| Расширения | Результат |
|---|---|
| `.csv`, `.xlsx`, `.xls`, `.parquet`, `.db`, `.sqlite` | `"structured"` |
| `.json`, `.jsonl`, `.xml`, `.html` | `"semi-structured"` |
| все остальные (`.txt`, `.pdf`, `.jpg`, `.mp3`, …) | `"unstructured"` |

Затем обойдите **всю** папку `data/` вместе с подпапками (`os.walk`) и посчитайте, сколько файлов каждого вида: `counts` — словарь `{"structured": ..., "semi-structured": ..., "unstructured": ...}`.

<details><summary>💡 Подсказка</summary>

В функции: `if ext in STRUCTURED: return "structured"`, затем проверка SEMI, иначе `"unstructured"`. В цикле: `counts[classify(name)] += 1`.

</details>

In [ ]:
import os

STRUCTURED = {".csv", ".xlsx", ".xls", ".parquet", ".db", ".sqlite"}
SEMI = {".json", ".jsonl", ".xml", ".html"}

def classify(filename):
    ext = os.path.splitext(filename)[1].lower()
    ...

counts = {"structured": 0, "semi-structured": 0, "unstructured": 0}
for root, dirs, files in os.walk("data"):
    for name in files:
        ...

print(classify("students.csv"), classify("books.json"), classify("uz_01.txt"))
print(counts)

## Самостоятельная работа

## Самостоятельная работа: «Изучение возможностей библиотеки Pandas»

1. Прочитайте `data/spark/students.jsonl` (формат JSON Lines) функцией `pd.read_json(..., lines=True)`. Чем этот формат удобнее обычного JSON для больших данных?
2. Сохраните таблицу `lib` из задания 4 обратно в XML методом `DataFrame.to_xml` (корень `library`, строки `book`) и сравните с исходным файлом.
3. Для всех текстов корпуса посчитайте долю слов с апострофом (`oʻ`, `gʻ`, `ʼ`) и добавьте этот признак в таблицу `corpus`.